# Exploring a recorded benchmark run

Every directory beside this notebook is one complete run of the
`amber-cas-bench` suite, exported with `amber-cas-bench publish`. The
Markdown and SVG in those directories are readable without running
anything; this notebook is for going further — slicing the CSV, comparing
scenarios, and checking a claim against the raw samples.

Run it with the environment this repository pins:

```
nix develop --command jupyter lab results/explore.ipynb
```

Two rules it enforces:

* **A run that declared itself invalid is not compared.** The suite marks a
  run invalid when an operation failed, a correctness check failed, a
  cross-backend check failed, or a requested backend could not run at all.
  Timings from such a run are diagnostics, not measurements, and every
  comparison below refuses to use them.
* **An operation a backend cannot perform is never a zero.** It is listed,
  with the reason, in its own section.

Nothing here is a widget: every result is printed text or a static plot, so
a saved copy of this notebook says the same thing as a live one.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 90)
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["figure.autolayout"] = True


def find_results_dir():
    """The directory holding the published runs.

    Set AMBER_BENCH_RESULTS to point somewhere else; otherwise this looks
    for the published runs beside the notebook and then under ./results.
    """
    override = os.environ.get("AMBER_BENCH_RESULTS")
    if override:
        return Path(override).resolve()
    for candidate in (Path.cwd(), Path.cwd() / "results", Path.cwd().parent / "results"):
        if candidate.is_dir() and any(candidate.glob("*/run.json")):
            return candidate.resolve()
    raise FileNotFoundError(
        "no published runs found. Set AMBER_BENCH_RESULTS to the results "
        "directory, or start the notebook from it."
    )


RESULTS = find_results_dir()
RUNS = sorted(
    (json.loads(p.read_text()) for p in RESULTS.glob("*/run.json")),
    key=lambda e: e["started_utc"],
    reverse=True,
)
print(f"results directory: {RESULTS}")
print(f"{len(RUNS)} recorded run(s):\n")
print(
    pd.DataFrame(
        [
            {
                "id": e["id"],
                "started (UTC)": e["started_utc"],
                "profile": e["profile"],
                "repeats": e["repeats"],
                "valid": e["valid"],
                "host": f"{e['host_cores']} x {e['host_cpu']}",
            }
            for e in RUNS
        ]
    ).to_string(index=False)
)

## Choosing a run

Set `AMBER_BENCH_RUN` to an `id` from the table above to pick one;
otherwise the most recent is used. Runs are **not** comparable with each
other: different profiles, hosts and configurations produce numbers that
mean different things, so everything below stays inside one run.

In [ ]:
wanted = os.environ.get("AMBER_BENCH_RUN")
if wanted:
    matches = [e for e in RUNS if e["id"] == wanted]
    if not matches:
        raise KeyError(f"no recorded run with id {wanted!r}")
    RUN = matches[0]
else:
    RUN = RUNS[0]

RUN_DIR = RESULTS / RUN["id"]
REPORT = json.loads((RUN_DIR / "report.json").read_text())

print(f"run:        {RUN['id']}")
print(f"started:    {RUN['started_utc']}")
print(f"profile:    {REPORT['profile']['name']} — {REPORT['profile']['description']}")
print(f"repeats:    {REPORT['repeats']} per (scenario, backend)")
print(f"seed:       {REPORT['seed']} (backend order seed {REPORT['order_seed']})")
print(f"host:       {REPORT['host']['cpu_model']} × {REPORT['host']['cpu_logical_cores']}")
print(f"harness:    {REPORT['harness_version']} commit {REPORT.get('harness_commit')}"
      f"{' (DIRTY working tree)' if REPORT.get('harness_dirty') else ''}")
print("\ncores measured:")
for name, rev in RUN["core_revisions"].items():
    print(f"  {name}: {rev}")
print(f"\ncache policy: {REPORT['host']['cache_policy']}")
print(f"\nstatistics:   {REPORT['statistics_method']}")

## The validity gate

This is the check that decides whether anything below may be read as a
measurement.

In [ ]:
class InvalidRunError(RuntimeError):
    """Raised when a comparison is attempted on a run that is not valid."""


def require_valid(run):
    """Refuses to compare a run whose own verdict is invalid."""
    if not run["valid"]:
        raise InvalidRunError(
            f"run {run['id']} declared itself INVALID "
            f"({'; '.join(run['invalid_reasons']) or 'no reason recorded'}). "
            "Its timings are diagnostics, not measurements, and this "
            "notebook will not rank them."
        )
    return True


VALID = RUN["valid"]
if VALID:
    require_valid(RUN)
    print(f"run {RUN['id']} is valid: comparisons below are measurements.")
else:
    print(f"run {RUN['id']} is INVALID. Reasons:")
    for reason in RUN["invalid_reasons"]:
        print(f"  * {reason}")
    print(
        "\nEvery comparison below is skipped. The failure tables at the end "
        "still run: that is what an invalid run is good for."
    )

## What was measured

`summary.csv` is long form: one row per (scenario, backend, operation,
metric). Only repetitions that were healthy throughout contribute to it;
`excluded_invalid_samples` counts successful timings that were deliberately
left out because something else in the same repetition failed.

In [ ]:
summary = pd.read_csv(RUN_DIR / "summary.csv")
samples = pd.read_csv(RUN_DIR / "samples.csv")
counters = pd.read_csv(RUN_DIR / "counters.csv")
checks = pd.read_csv(RUN_DIR / "verifications.csv")

coverage = (
    summary[summary.metric == "wall_ns"]
    .groupby(["group", "scenario"])
    .agg(
        backends=("backend", lambda s: ", ".join(sorted(set(s)))),
        operations=("op", lambda s: len(set(s))),
    )
    .reset_index()
)
print(coverage.to_string(index=False))
print(f"\n{len(samples)} raw operation sample(s), {len(checks)} correctness check(s)")

In [ ]:
def metric_table(scenario, metric="wall_ns"):
    """Median of one metric for one scenario, backends as columns.

    Rows with no valid repetition are dropped rather than shown as zero;
    what happened to them is in the `no measurement` section below.
    """
    rows = summary[
        (summary.scenario == scenario) & (summary.metric == metric) & (summary.ok > 0)
    ]
    if rows.empty:
        return None
    table = rows.pivot_table(
        index="op", columns="backend", values="median", aggfunc="first", sort=False
    )
    return table.reindex(rows.op.drop_duplicates())


def compare(scenario, metric="wall_ns", scale=1e6, unit="ms", title=None):
    """Prints and plots one metric of one scenario. One scenario only."""
    require_valid(RUN)
    table = metric_table(scenario, metric)
    if table is None:
        print(f"{scenario}: nothing measured for {metric}")
        return
    scaled = table / scale
    print(f"\n=== {scenario} — {title or metric} ({unit}) ===")
    print(scaled.round(2).to_string())

    ax = scaled.plot.barh(figsize=(10, 0.45 * scaled.size + 2))
    ax.set_xlabel(f"{title or metric} [{unit}] — median of {REPORT['repeats']} repetition(s)")
    ax.set_ylabel("")
    ax.set_title(f"{scenario} — {title or metric}")
    ax.invert_yaxis()
    ax.legend(fontsize=8, loc="lower right")
    plt.show()


SCENARIOS = list(dict.fromkeys(summary.scenario))
print("scenarios in this run:", ", ".join(SCENARIOS))

## Elapsed time, scenario by scenario

A bar is only comparable with the other bars of its own group: those are
the same operation over the same bytes. Across scenarios nothing is
comparable at all.

In [ ]:
if VALID:
    for scenario in SCENARIOS:
        compare(scenario, "wall_ns", 1e6, "ms", "elapsed time")
else:
    print("skipped: this run is invalid.")

## What each backend left on disk

Allocated bytes of the backend's own store directory after each operation —
what the filesystem charges, not the sum of file lengths.

In [ ]:
if VALID:
    for scenario in SCENARIOS:
        if metric_table(scenario, "store_allocated_bytes") is not None:
            compare(scenario, "store_allocated_bytes", 1 << 20, "MiB", "store size")
else:
    print("skipped: this run is invalid.")

## Object storage: bytes and requests

Counted at the socket by the measuring gateway, in both directions,
retries included. `retention_cleanup` is the one operation where the
backends are not asked for the same thing — see `retained_references` in
`counters.csv` and the notes in `REPORT.md` — so it is not a ranking.

In [ ]:
def counter_table(scenario, counter):
    rows = counters[(counters.scenario == scenario) & (counters.counter == counter)]
    if rows.empty:
        return None
    return rows.pivot_table(
        index="op", columns="backend", values="value", aggfunc="median", sort=False
    )


if VALID:
    blob_scenarios = [s for s in SCENARIOS if s.startswith("blob/")]
    for scenario in blob_scenarios:
        for counter, unit, scale in [
            ("s3_bytes_up", "MiB uploaded", 1 << 20),
            ("s3_bytes_down", "MiB downloaded", 1 << 20),
            ("s3_requests_total", "HTTP requests", 1),
        ]:
            table = counter_table(scenario, counter)
            if table is None:
                continue
            print(f"\n=== {scenario} — {counter} ({unit}) ===")
            print((table / scale).round(3).to_string())
        delivered = counter_table(scenario, "delivered_references")
        if delivered is not None:
            print(f"\n--- {scenario}: references each transfer delivered ---")
            print(delivered.astype("Int64").to_string())
    if not blob_scenarios:
        print("this run has no object-storage scenario.")
else:
    print("skipped: this run is invalid.")

## Where there is no number

Operations a backend cannot perform, and operations that failed. These are
never zeros and never blank rows: the reason is recorded with them.

In [ ]:
gaps = summary[(summary.metric == "wall_ns") & (summary.ok == 0)][
    ["scenario", "backend", "op", "unsupported", "failed", "excluded_invalid_samples", "reason"]
]
if gaps.empty:
    print("every backend produced a measurement for every operation it was asked for.")
else:
    print(gaps.to_string(index=False))

excluded = summary[summary.excluded_invalid_samples > 0]
if not excluded.empty:
    print("\nsuccessful timings excluded from the statistics (their repetition was not healthy):")
    print(
        excluded[["scenario", "backend", "op", "excluded_invalid_samples"]]
        .drop_duplicates()
        .to_string(index=False)
    )

## Correctness

Every scenario verifies what it stored against an independent manifest.
These checks are what make the timings mean anything.

In [ ]:
print(f"{len(checks)} check(s), {int((~checks.passed).sum())} failed")
print()
print(
    checks.groupby(["scenario", "check"])
    .agg(runs=("passed", "size"), failed=("passed", lambda s: int((~s).sum())))
    .reset_index()
    .to_string(index=False)
)
failed_checks = checks[~checks.passed]
if not failed_checks.empty:
    print("\nFAILED:")
    print(failed_checks[["scenario", "backend", "rep", "check", "detail"]].to_string(index=False))

## The gate, demonstrated

The refusal above is not decoration. Asking for a comparison of a run
marked invalid raises, wherever it is asked from.

In [ ]:
pretend_invalid = dict(RUN, valid=False, invalid_reasons=["1 recorded failure(s)"])
try:
    require_valid(pretend_invalid)
except InvalidRunError as e:
    print("refused, as it should be:\n ", e)
else:
    raise AssertionError("the validity gate did not refuse an invalid run")

## Reading these numbers

* **Within a scenario, within a run.** Two bars of one group describe the
  same bytes and the same operation. Nothing else here is a comparison.
* **Small samples.** The profiles used here repeat each pair a handful of
  times. `summary.csv` carries `cv`, `stddev` and `p95`; `samples.csv`
  carries every raw sample. A difference of the same order as the spread is
  indicative only.
* **Warm cache unless stated.** Read and restore numbers follow the writes
  that filled the store. The run's `cache_policy`, printed above, says
  exactly what was and was not done.
* **Unlike guarantees.** restic encrypts every byte and the Amber cores do
  not; Git records commit identity and parentage and an Amber reference
  does not; a Nix store deduplicates by whole path. `REPORT.md` states each
  backend's compression, encryption, durability and concurrency next to its
  numbers, and none of these results imply a Nix daemon or a Git client
  could be replaced wholesale — they are storage-layer comparisons.